<a href="https://colab.research.google.com/github/zergood/path-to-master-ai-ops/blob/main/week-01/day-00-setup/first_kernel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numba
import numpy as np

In [1]:
import torch

In [42]:
import numpy as np
from numba import cuda

start = cuda.event()
end = cuda.event()

@cuda.jit
def map_kernel(out, a):
    i = cuda.grid(1)
    if i < a.shape[0]:
        out[i] = a[i] + 10

a = np.arange(10_000_000, dtype=np.float32)
out = np.zeros_like(a)
d_a = cuda.to_device(a)
d_out = cuda.device_array_like(d_a)

threads = 256
blocks = (a.size + threads - 1) // threads

for _ in range(10):                               # прогрев + JIT
    map_kernel[blocks, threads](d_out, d_a)
cuda.synchronize()

start.record()
map_kernel[blocks, threads](d_out, d_a)   # [блоков, потоков в блоке]
end.record()
end.synchronize()

print(start.elapsed_time(end))


0.5086399912834167


In [39]:
a = torch.arange(10_000_000, device="cuda", dtype=torch.float32)
out_torch = torch.empty_like(a)

for _ in range(10):                       # прогрев
    torch.add(a, 10, out=out_torch)
torch.cuda.synchronize()

s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
iters = 100
s.record()
for _ in range(iters):
    torch.add(a, 10, out=out_torch)
e.record()
torch.cuda.synchronize()

ms = s.elapsed_time(e) / iters
gb_s = 2 * a.numel() * a.element_size() / ms / 1e6
print(f"{ms:.4f} ms, {gb_s:.0f} GB/s")

0.3486 ms, 230 GB/s
